# Base39 Encoder/Decoder Evaluation

Test the base39encoder round-trip (encode → decode) and measure reconstruction errors for pendulum simulator data. Verify that tracking error stays within 1-2 counts as specified.

## Section 1: Import Required Libraries and Modules

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sim import PendulumSimulator
from base39encoder import Base39Encoder, Base39Decoder

print("✓ Imports successful")
print(f"Base39Encoder: {Base39Encoder}")
print(f"Base39Decoder: {Base39Decoder}")

✓ Imports successful
Base39Encoder: <class 'base39encoder.Base39Encoder'>
Base39Decoder: <class 'base39encoder.Base39Decoder'>


## Section 2: Load and Inspect Base39Encoder

Initialize encoder/decoder and understand the encoding mechanism.

In [2]:
# Generate pendulum simulator data
sim = PendulumSimulator(sample_rate_hz=5000, noise_rms_lsb=0.8)
final_x, final_y, angles, rates, impact_idx = sim.generate_data()

print(f"Generated {len(final_x)} samples")
print(f"X (cos) range: [{final_x.min()}, {final_x.max()}]")
print(f"Y (sin) range: [{final_y.min()}, {final_y.max()}]")
print(f"Impact at sample {impact_idx}")

# Initialize encoder and decoder
encoder = Base39Encoder()
decoder = Base39Decoder()

print(f"\nEncoder initial state:")
print(f"  cos_state={encoder.cos_state}, sin_state={encoder.sin_state}")
print(f"  filtered_omega={encoder.filtered_omega}, filtered_dr={encoder.filtered_dr}")

print(f"\nDecoder initial state:")
print(f"  is_synchronized={decoder.is_synchronized}")

Generated 12097 samples
X (cos) range: [-2049, 2049]
Y (sin) range: [-2049, 2049]
Impact at sample 8305

Encoder initial state:
  cos_state=1500, sin_state=0
  filtered_omega=0, filtered_dr=0

Decoder initial state:
  is_synchronized=False


## Section 3: Simulate Encoding and Decoding Process

Encode all samples and decode them back to measure round-trip error.

In [3]:
print("="*70)
print("ENCODING PROCESS")
print("="*70)

# Encode all samples
byte_stream = b""
sample_count = 0
for i in range(len(final_x)):
    cos_val = int(final_x[i])
    sin_val = int(final_y[i])
    
    encoded_bytes = encoder.encode_sample(cos_val, sin_val)
    if encoded_bytes:
        byte_stream += encoded_bytes
        sample_count += 1

print(f"\nEncoded {len(final_x)} samples into {len(byte_stream)} bytes")
print(f"Compression ratio: {len(final_x) * 4 / len(byte_stream):.2f}× (4 bytes per sample → {len(byte_stream)} bytes)")
print(f"Bits per sample: {len(byte_stream) * 8 / len(final_x):.2f}")

print("\n" + "="*70)
print("DECODING PROCESS")
print("="*70)

# Decode the byte stream
decoded_positions = decoder.parse_bytes(byte_stream)
print(f"\nDecoded {len(decoded_positions)} (cos, sin) pairs from byte stream")

# Show first few decoded samples for sanity check
print(f"\nFirst 10 decoded samples (cos, sin):")
for i, (cos_dec, sin_dec) in enumerate(decoded_positions[:10]):
    print(f"  [{i:3d}] cos={cos_dec:6d}, sin={sin_dec:6d}")

print(f"\nLast 10 decoded samples (cos, sin):")
for i, (cos_dec, sin_dec) in enumerate(decoded_positions[-10:], start=len(decoded_positions)-10):
    print(f"  [{i:3d}] cos={cos_dec:6d}, sin={sin_dec:6d}")

ENCODING PROCESS

Encoded 12097 samples into 8224 bytes
Compression ratio: 5.88× (4 bytes per sample → 8224 bytes)
Bits per sample: 5.44

DECODING PROCESS

Decoded 8454 (cos, sin) pairs from byte stream

First 10 decoded samples (cos, sin):
  [  0] cos=     7, sin= -2048
  [  1] cos=-92166, sin=  3238
  [  2] cos=147588, sin=1663607
  [  3] cos=9267059, sin=-1519632
  [  4] cos=-480318, sin=-1768440
  [  5] cos=3151755, sin=-1359866
  [  6] cos=4704271, sin=6824901
  [  7] cos=-12390447, sin=14768906
  [  8] cos=-36942435, sin=-12740792
  [  9] cos=-10423962, sin=-65900494

Last 10 decoded samples (cos, sin):
  [8444] cos=  -164, sin=  1744
  [8445] cos=  -166, sin=  1744
  [8446] cos=  -168, sin=  1744
  [8447] cos=  -170, sin=  1744
  [8448] cos=  -172, sin=  1749
  [8449] cos=  -174, sin=  1751
  [8450] cos=  -176, sin=  1753
  [8451] cos=  -178, sin=  1753
  [8452] cos=  -180, sin=  1752
  [8453] cos=    -1, sin=  2047


## Section 4: Compare Original vs Decoded Values

In [6]:
print("="*70)
print("ERROR ANALYSIS: Original vs Decoded")
print("="*70)

# Align original and decoded data
n_decoded = len(decoded_positions)
n_original = len(final_x)

print(f"\nSample count comparison:")
print(f"  Original samples: {n_original}")
print(f"  Decoded samples: {n_decoded}")

# Check range of decoded values
decoded_cos_vals = np.array([pos[0] for pos in decoded_positions])
decoded_sin_vals = np.array([pos[1] for pos in decoded_positions])

print(f"\nDecoded value ranges (all samples):")
print(f"  Cos min: {np.min(decoded_cos_vals)}, max: {np.max(decoded_cos_vals)}")
print(f"  Sin min: {np.min(decoded_sin_vals)}, max: {np.max(decoded_sin_vals)}")

# Find first synchronized sample more carefully
# Look for samples where both cos and sin are close to reasonable range AND appear stable
first_valid_idx = -1
for i in range(len(decoded_positions) - 10):
    # Check this sample and next 10
    window = decoded_cos_vals[i:i+10]
    if np.all(np.abs(window) <= 2500):  # All values in a reasonable window?
        first_valid_idx = i
        break

if first_valid_idx < 0:
    print("WARNING: Could not find synchronized region in decoded data")
    print("Samples may not have had proper sync frames emitted by encoder")
    print("\nShowing first 20 decoded samples:")
    for i in range(min(20, len(decoded_positions))):
        cos_dec, sin_dec = decoded_positions[i]
        print(f"  [{i:3d}] cos={cos_dec:12d}, sin={sin_dec:12d}")
else:
    print(f"\nFirst synchronized region found at index {first_valid_idx}")
    
    # Compute errors (only for synchronized samples)
    n_compare = min(n_original - first_valid_idx, n_decoded - first_valid_idx)
    original_cos = final_x[first_valid_idx:first_valid_idx+n_compare].astype(np.int32)
    original_sin = final_y[first_valid_idx:first_valid_idx+n_compare].astype(np.int32)
    decoded_cos = np.array([pos[0] for pos in decoded_positions[first_valid_idx:first_valid_idx+n_compare]], dtype=np.int64)
    decoded_sin = np.array([pos[1] for pos in decoded_positions[first_valid_idx:first_valid_idx+n_compare]], dtype=np.int64)

    err_cos = decoded_cos - original_cos
    err_sin = decoded_sin - original_sin
    err_magnitude = np.sqrt(err_cos**2 + err_sin**2)

    print(f"Analyzing {n_compare} synchronized samples (starting from index {first_valid_idx})")

    print(f"\nCosine (X) errors:")
    print(f"  Mean: {np.mean(err_cos):.3f} LSB")
    print(f"  Std: {np.std(err_cos):.3f} LSB")
    print(f"  Min: {np.min(err_cos)} LSB")
    print(f"  Max: {np.max(err_cos)} LSB")
    print(f"  |Error| ≤ 1: {100*np.mean(np.abs(err_cos) <= 1):.1f}%")
    print(f"  |Error| ≤ 2: {100*np.mean(np.abs(err_cos) <= 2):.1f}%")

    print(f"\nSine (Y) errors:")
    print(f"  Mean: {np.mean(err_sin):.3f} LSB")
    print(f"  Std: {np.std(err_sin):.3f} LSB")
    print(f"  Min: {np.min(err_sin)} LSB")
    print(f"  Max: {np.max(err_sin)} LSB")
    print(f"  |Error| ≤ 1: {100*np.mean(np.abs(err_sin) <= 1):.1f}%")
    print(f"  |Error| ≤ 2: {100*np.mean(np.abs(err_sin) <= 2):.1f}%")

    print(f"\nMagnitude errors (sqrt(err_cos² + err_sin²)):")
    print(f"  Mean: {np.mean(err_magnitude):.3f} LSB")
    print(f"  Std: {np.std(err_magnitude):.3f} LSB")
    print(f"  Min: {np.min(err_magnitude):.3f} LSB")
    print(f"  Max: {np.max(err_magnitude):.3f} LSB")
    print(f"  |Error| ≤ 1: {100*np.mean(err_magnitude <= 1):.1f}%")
    print(f"  |Error| ≤ 2: {100*np.mean(err_magnitude <= 2):.1f}%")
    print(f"  |Error| ≤ 3: {100*np.mean(err_magnitude <= 3):.1f}%")

ERROR ANALYSIS: Original vs Decoded

Sample count comparison:
  Original samples: 12097
  Decoded samples: 8454

Decoded value ranges (all samples):
  Cos min: -98200128084, max: 53341674002
  Sin min: -118195598885, max: 88338184570

First synchronized region found at index 6118
Analyzing 2336 synchronized samples (starting from index 6118)

Cosine (X) errors:
  Mean: -930.881 LSB
  Std: 3818.619 LSB
  Min: -24268 LSB
  Max: 160690 LSB
  |Error| ≤ 1: 0.0%
  |Error| ≤ 2: 0.0%

Sine (Y) errors:
  Mean: 1089.274 LSB
  Std: 3636.135 LSB
  Min: -112807 LSB
  Max: 115392 LSB
  |Error| ≤ 1: 0.1%
  |Error| ≤ 2: 0.2%

Magnitude errors (sqrt(err_cos² + err_sin²)):
  Mean: 2470.564 LSB
  Std: 4873.674 LSB
  Min: 118.343 LSB
  Max: 196333.123 LSB
  |Error| ≤ 1: 0.0%
  |Error| ≤ 2: 0.0%
  |Error| ≤ 3: 0.0%


## Detailed Debugging: Encoder/Decoder Flow

Something is fundamentally wrong—errors are in the thousands of LSB, not 1-2. Let me investigate the issue.

In [7]:
print("Checking first 50 sync frames generated:")
print(f"  Total byte stream: {len(byte_stream)} bytes")
print(f"  Approx sync frames: {len(byte_stream) // 6:.0f} (each ~6 bytes)")

# Check byte patterns for sync frames (0xF0 or 0xF1 patterns)
from base39encoder import SYNC_BASE_CODE, SYNC_MAX_CODE
import struct

sync_frame_count = 0
sync_indices = []
for i in range(0, len(byte_stream)-5):
    try:
        w1 = struct.unpack(">H", byte_stream[i:i+2])[0]
        if SYNC_BASE_CODE <= w1 <= SYNC_MAX_CODE:
            sync_frame_count += 1
            sync_indices.append(i)
            if sync_frame_count <= 10:
                print(f"  Sync frame {sync_frame_count} at byte offset {i}, w1=0x{w1:04x}")
    except:
        pass

print(f"\nTotal sync frames found in byte stream: {sync_frame_count}")
print(f"First sync frame location (byte offset): {sync_indices[0] if sync_indices else 'NONE'}")
print(f"\nDecoder stats:")
print(f"  Decoded samples: {n_decoded}")
print(f"  First valid sample: index {first_valid_idx if first_valid_idx >= 0 else 'NONE'}")
print(f"\nImplication:")
if first_valid_idx > n_decoded * 0.5:
    print("  ⚠ Decoder took >50% of samples to synchronize - something is wrong with sync frame generation or detection!")

Checking first 50 sync frames generated:
  Total byte stream: 8224 bytes
  Approx sync frames: 1370 (each ~6 bytes)
  Sync frame 1 at byte offset 37, w1=0xf1da
  Sync frame 2 at byte offset 63, w1=0xf2da
  Sync frame 3 at byte offset 67, w1=0xe8cf
  Sync frame 4 at byte offset 71, w1=0xe9c3
  Sync frame 5 at byte offset 77, w1=0xefb7
  Sync frame 6 at byte offset 89, w1=0xe9c3
  Sync frame 7 at byte offset 93, w1=0xede0
  Sync frame 8 at byte offset 97, w1=0xf4c3
  Sync frame 9 at byte offset 107, w1=0xf1da
  Sync frame 10 at byte offset 119, w1=0xf3c9

Total sync frames found in byte stream: 97
First sync frame location (byte offset): 37

Decoder stats:
  Decoded samples: 8454
  First valid sample: index 6118

Implication:
  ⚠ Decoder took >50% of samples to synchronize - something is wrong with sync frame generation or detection!


## Summary: Base39 Encoder/Decoder Issue Report

**Status:** ⚠️ ERRORS EXCEED SPECIFICATION BY >1000×

### Key Findings:
1. **Sync frames:** 97 sync frames detected in byte stream (byte offset 37-onward)
2. **Decoder synchronization:** Does NOT synchronize until sample 6118 / 8454 (72% through stream)
3. **Error magnitude:** After sync, errors are ~2500 LSB mean (required: 1-2 LSB)

### Root Cause Analysis:
The encoder/decoder appears to have a fundamental issue with:
- Initial synchronization detection from unsync state
- Proper alignment of decoded samples with original samples
- The decoder's `parse_bytes()` function may not be correctly processing the byte stream

### Recommendation:
The encoder/decoder implementation needs review. Current issues:
1. Decoder is losing synchronization for first ~72% of stream
2. Even after sync, errors are 1000× larger than specification
3. This suggests either:
   - Incorrect byte ordering or packing
   - Misalignment between encoder state and decoder state
   - Malformed sync frame generation or detection

**Next steps:** Verify encoder/decoder design with original designer.

## Section 5: Calculate and Visualize Error Metrics

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Error scatter plot
axes[0, 0].scatter(err_cos, err_sin, alpha=0.3, s=1)
axes[0, 0].set_xlim(-3, 3)
axes[0, 0].set_ylim(-3, 3)
axes[0, 0].set_title("Error Scatter (cos_err vs sin_err)")
axes[0, 0].set_xlabel("Cos Error (LSB)")
axes[0, 0].set_ylabel("Sin Error (LSB)")
axes[0, 0].axhline(0, color='k', linestyle='-', linewidth=0.5)
axes[0, 0].axvline(0, color='k', linestyle='-', linewidth=0.5)
axes[0, 0].grid(True, alpha=0.3)

# Cos error histogram
axes[0, 1].hist(err_cos, bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title("Cosine Error Distribution")
axes[0, 1].set_xlabel("Error (LSB)")
axes[0, 1].set_ylabel("Frequency")
axes[0, 1].axvline(np.mean(err_cos), color='r', linestyle='--', linewidth=2, label=f'Mean={np.mean(err_cos):.2f}')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Sin error histogram
axes[0, 2].hist(err_sin, bins=30, edgecolor='black', alpha=0.7, color='orange')
axes[0, 2].set_title("Sine Error Distribution")
axes[0, 2].set_xlabel("Error (LSB)")
axes[0, 2].set_ylabel("Frequency")
axes[0, 2].axvline(np.mean(err_sin), color='r', linestyle='--', linewidth=2, label=f'Mean={np.mean(err_sin):.2f}')
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# Error time series (first 2000 samples)
sample_range = slice(0, min(2000, len(err_cos)))
axes[1, 0].plot(err_cos[sample_range], alpha=0.6, linewidth=0.5, label='Cos Error')
axes[1, 0].plot(err_sin[sample_range], alpha=0.6, linewidth=0.5, label='Sin Error')
axes[1, 0].set_title("Error Time Series (First 2000 Samples)")
axes[1, 0].set_ylabel("Error (LSB)")
axes[1, 0].set_xlabel("Sample")
axes[1, 0].axhline(0, color='k', linestyle='-', linewidth=0.5)
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Magnitude error histogram
axes[1, 1].hist(err_magnitude, bins=50, edgecolor='black', alpha=0.7, color='green')
axes[1, 1].set_title("Magnitude Error Distribution")
axes[1, 1].set_xlabel("Error (LSB)")
axes[1, 1].set_ylabel("Frequency")
axes[1, 1].axvline(1, color='orange', linestyle='--', linewidth=1.5, label='Threshold=1')
axes[1, 1].axvline(2, color='red', linestyle='--', linewidth=1.5, label='Threshold=2')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# Cumulative distribution
sorted_errors = np.sort(err_magnitude)
cdf = np.arange(1, len(sorted_errors) + 1) / len(sorted_errors)
axes[1, 2].plot(sorted_errors, cdf, linewidth=2)
axes[1, 2].axvline(1, color='orange', linestyle='--', linewidth=1.5, label='1 LSB')
axes[1, 2].axvline(2, color='red', linestyle='--', linewidth=1.5, label='2 LSB')
axes[1, 2].set_title("Cumulative Error Distribution")
axes[1, 2].set_xlabel("Error Magnitude (LSB)")
axes[1, 2].set_ylabel("CDF")
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('base39_error_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("Plot saved to base39_error_analysis.png")

## Section 6: Validate Error Tracking Range

In [ ]:
print("="*70)
print("VALIDATION: Tracking Error Summary")
print("="*70)

# Check requirement: track within 1-2 counts
within_1_cos = np.sum(np.abs(err_cos) <= 1)
within_2_cos = np.sum(np.abs(err_cos) <= 2)
within_1_sin = np.sum(np.abs(err_sin) <= 1)
within_2_sin = np.sum(np.abs(err_sin) <= 2)
within_1_mag = np.sum(err_magnitude <= 1)
within_2_mag = np.sum(err_magnitude <= 2)

total = len(err_cos)

print(f"\nCosine Error Coverage:")
print(f"  Within ±1 LSB: {within_1_cos}/{total} ({100*within_1_cos/total:.1f}%)")
print(f"  Within ±2 LSB: {within_2_cos}/{total} ({100*within_2_cos/total:.1f}%)")
print(f"  Outside ±2 LSB: {total - within_2_cos}/{total} ({100*(total-within_2_cos)/total:.2f}%)")

print(f"\nSine Error Coverage:")
print(f"  Within ±1 LSB: {within_1_sin}/{total} ({100*within_1_sin/total:.1f}%)")
print(f"  Within ±2 LSB: {within_2_sin}/{total} ({100*within_2_sin/total:.1f}%)")
print(f"  Outside ±2 LSB: {total - within_2_sin}/{total} ({100*(total-within_2_sin)/total:.2f}%)")

print(f"\nMagnitude Error Coverage:")
print(f"  Within 1 LSB: {within_1_mag}/{total} ({100*within_1_mag/total:.1f}%)")
print(f"  Within 2 LSB: {within_2_mag}/{total} ({100*within_2_mag/total:.1f}%)")
print(f"  Outside 2 LSB: {total - within_2_mag}/{total} ({100*(total-within_2_mag)/total:.2f}%)")

print(f"\n{'='*70}")
if (100*(total-within_2_cos)/total < 1.0 and 
    100*(total-within_2_sin)/total < 1.0 and 
    100*(total-within_2_mag)/total < 1.0):
    print("✓ PASS: All errors track within 1-2 counts as specified!")
else:
    print("⚠ WARNING: Some errors exceed 2 LSB threshold")
    print(f"  Cos outliers: {100*(total-within_2_cos)/total:.2f}%")
    print(f"  Sin outliers: {100*(total-within_2_sin)/total:.2f}%")
    print(f"  Mag outliers: {100*(total-within_2_mag)/total:.2f}%")

print(f"{'='*70}")

# Find and display outliers
outliers_cos = np.where(np.abs(err_cos) > 2)[0]
outliers_sin = np.where(np.abs(err_sin) > 2)[0]
outliers_mag = np.where(err_magnitude > 2)[0]

if len(outliers_cos) > 0:
    print(f"\nCosine Error Outliers (|err| > 2 LSB): {len(outliers_cos)} samples")
    for idx in outliers_cos[:10]:  # Show first 10
        print(f"  Sample {idx}: original={original_cos[idx]}, decoded={decoded_cos[idx]}, error={err_cos[idx]}")
    if len(outliers_cos) > 10:
        print(f"  ... and {len(outliers_cos)-10} more")

if len(outliers_sin) > 0:
    print(f"\nSine Error Outliers (|err| > 2 LSB): {len(outliers_sin)} samples")
    for idx in outliers_sin[:10]:  # Show first 10
        print(f"  Sample {idx}: original={original_sin[idx]}, decoded={decoded_sin[idx]}, error={err_sin[idx]}")
    if len(outliers_sin) > 10:
        print(f"  ... and {len(outliers_sin)-10} more")

if len(outliers_mag) > 0:
    print(f"\nMagnitude Error Outliers (err > 2 LSB): {len(outliers_mag)} samples")
    for idx in outliers_mag[:10]:  # Show first 10
        print(f"  Sample {idx}: cos_err={err_cos[idx]}, sin_err={err_sin[idx]}, magnitude={err_magnitude[idx]:.2f}")
    if len(outliers_mag) > 10:
        print(f"  ... and {len(outliers_mag)-10} more")